# Detector de Baches - Procesamiento con GPU
Procesa el video completo sin saltarse frames. Sube el video a Google Drive antes de correr.

In [ ]:
# ── Paso 1: Instalar dependencias ──────────────────────────
!pip install ultralytics huggingface_hub -q
import ultralytics
ultralytics.checks()

In [ ]:
# ── Paso 2: Montar Google Drive ────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# CAMBIA este nombre al de tu video en Drive
VIDEO_ENTRADA = '/content/drive/MyDrive/holes.mp4'
VIDEO_SALIDA  = '/content/drive/MyDrive/holes_baches_resultado.mp4'
CARPETA_IMGS  = '/content/drive/MyDrive/baches_detectados/'

import os
os.makedirs(CARPETA_IMGS, exist_ok=True)
print('Drive montado.')
print(f'Video entrada : {VIDEO_ENTRADA}')
print(f'Video salida  : {VIDEO_SALIDA}')
print(f'Imágenes      : {CARPETA_IMGS}')

In [ ]:
# ── Paso 3: Descargar modelo ───────────────────────────────
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

modelo_path = hf_hub_download(
    repo_id='keremberke/yolov8m-pothole-segmentation',
    filename='best.pt'
)
modelo = YOLO(modelo_path)
print('Modelo cargado.')

In [ ]:
# ── Paso 4: Procesar video completo ───────────────────────
import cv2
import numpy as np
from pathlib import Path

CONFIANZA_MIN  = 0.45
IOU_NMS        = 0.45
ROI_INICIO     = 0.45   # ignorar parte superior del frame (cielo/edificios)
FRAMES_CONFIRM = 2      # frames consecutivos para guardar imagen del bache
VERDE_MAX      = 0.25

def es_bache_valido(frame, x1, y1, x2, y2):
    region = frame[y1:y2, x1:x2]
    if region.size == 0:
        return False
    hsv = cv2.cvtColor(region, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    mascara_verde = cv2.inRange(hsv, (25, 30, 30), (95, 255, 255))
    pct_verde = float(mascara_verde.sum()) / 255 / mascara_verde.size
    if pct_verde > VERDE_MAX:
        return False
    if float(s.mean()) > 60:
        return False
    return True

def color_conf(conf):
    if conf >= 0.80: return (0, 200, 0)
    if conf >= 0.65: return (0, 165, 255)
    return (0, 0, 255)

def guardar_recorte(frame, x1, y1, x2, y2, conf, carpeta, num):
    pad = 25
    h, w = frame.shape[:2]
    rx1, ry1 = max(0, x1-pad), max(0, y1-pad)
    rx2, ry2 = min(w, x2+pad), min(h, y2+pad)
    recorte = frame[ry1:ry2, rx1:rx2].copy()
    color = color_conf(conf)
    cv2.rectangle(recorte, (x1-rx1, y1-ry1), (x2-rx1, y2-ry1), color, 2)
    cv2.putText(recorte, f'Bache {conf:.0%}', (x1-rx1, max(y1-ry1-8, 14)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    nombre = f'bache_{num:04d}_conf{int(conf*100)}.jpg'
    cv2.imwrite(carpeta + nombre, recorte)
    return nombre

# ── Abrir video ────────────────────────────────────────────
cap   = cv2.VideoCapture(VIDEO_ENTRADA)
fps   = cap.get(cv2.CAP_PROP_FPS)
ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
roi_y = int(alto * ROI_INICIO)
print(f'Video: {ancho}x{alto} @ {fps:.1f} fps | {total} frames')

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(VIDEO_SALIDA, fourcc, fps, (ancho, alto))

tracked       = {}
img_guardadas = 0
frame_idx     = 0

print('Procesando...')
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f'  Frame {frame_idx}/{total}  |  Baches confirmados: {img_guardadas}')

    output_frame = frame.copy()
    roi_frame    = frame[roi_y:alto, 0:ancho]

    resultados = modelo.track(roi_frame, persist=True,
                              conf=CONFIANZA_MIN, iou=IOU_NMS, verbose=False)

    cv2.line(output_frame, (0, roi_y), (ancho, roi_y), (60, 60, 60), 1)

    if resultados and resultados[0].boxes is not None:
        boxes = resultados[0].boxes
        for i in range(len(boxes)):
            conf = float(boxes.conf[i])
            if conf < CONFIANZA_MIN:
                continue
            x1, y1, x2, y2 = boxes.xyxy[i].cpu().numpy().astype(int)
            y1 += roi_y; y2 += roi_y

            if not es_bache_valido(frame, x1, y1, x2, y2):
                continue

            track_id = int(boxes.id[i]) if boxes.id is not None else -(frame_idx*100+i)

            if track_id not in tracked:
                tracked[track_id] = {'frames_visto': 0, 'mejor_conf': 0.0,
                                     'mejor_frame': None, 'mejor_bbox': None, 'guardado': False}
            t = tracked[track_id]
            t['frames_visto'] += 1
            if conf > t['mejor_conf']:
                t['mejor_conf']  = conf
                t['mejor_frame'] = frame.copy()
                t['mejor_bbox']  = (x1, y1, x2, y2)

            if t['frames_visto'] >= FRAMES_CONFIRM and not t['guardado']:
                img_guardadas += 1
                bx1,by1,bx2,by2 = t['mejor_bbox']
                nombre = guardar_recorte(t['mejor_frame'], bx1,by1,bx2,by2,
                                         t['mejor_conf'], CARPETA_IMGS, img_guardadas)
                t['guardado'] = True
                print(f'  ✓ Bache ID {track_id}  conf={t["mejor_conf"]:.0%}  → {nombre}')

            color = color_conf(conf)
            cv2.rectangle(output_frame, (x1, y1), (x2, y2), color, 2)
            label = f'Bache #{track_id}  {conf:.0%}'
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(output_frame, (x1, y1-th-10), (x1+tw+6, y1), color, -1)
            cv2.putText(output_frame, label, (x1+3, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

            if resultados[0].masks is not None:
                try:
                    mask = resultados[0].masks.data[i].cpu().numpy()
                    mask_r = cv2.resize(mask, (ancho, alto-roi_y))
                    mb = mask_r > 0.5
                    overlay = output_frame.copy()
                    overlay[roi_y:alto,:][mb] = (overlay[roi_y:alto,:][mb]*0.5 + np.array(color)*0.5).astype(np.uint8)
                    output_frame = overlay
                except: pass

    cv2.rectangle(output_frame, (0,0), (340,60), (20,20,20), -1)
    cv2.putText(output_frame, f'Baches confirmados: {img_guardadas}',
                (10,40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (100,255,100), 2)
    writer.write(output_frame)

cap.release()
writer.release()
print(f'\n✓ Listo. Baches detectados: {img_guardadas}')
print(f'Video guardado en: {VIDEO_SALIDA}')
print(f'Imágenes en      : {CARPETA_IMGS}')